# Lab 09a · Which fields must a person check?

[Open in Colab](https://colab.research.google.com/github/AIGuruOne/OQ-labs/blob/main/notebooks/09a_vision_review_rule.ipynb)

**Day 3 · S16 · Step 2 of 2** · 20 min · One API key. No GPU, no Ollama.

## What you will be able to do after this

**Decide, field by field, what a model may write into a system of record unchecked —
and what must go to a person — by measuring it against documents you already know the
answers to.**

**You will know it landed** when you can finish this sentence for your own capstone
brief, with your own numbers:

> *"A model may write ______ unchecked. A person must see ______.
> That costs ______ reviews, and ______ wrong values still get through."*

**Where you use it:**
S27 capstone — deployment checklist row 3, *a wrong answer has a defined path*.
S29 governance pack — template 4, *approval points*.

---

Lab 08a showed you one form failing. This one asks the only question that follows:
**how often, on which fields, and who checks them?**

**A correct result looks like:** a wrong-field count out of 60, a per-field table where
the errors are **not** spread evenly, a rule you chose yourself, and the cost of that
rule — including the wrong values it still lets through.

> All data in this lab is synthetic. No real OQ material anywhere.

## 1. Setup

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai==3.0.0"], check=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/AIGuruOne/OQ-labs.git"], check=False)
    sys.path.insert(0, "OQ-labs/notebooks")
    import os
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))

import vision_lab as lab
print("ready |", lab.MODEL)

## 2. Ten work orders, photographed

Not the terrible photograph from 08a. These are **decent** phone photos, taken in good
light — the kind anyone would send you without apologising for it.

We drew all ten, so we know every value on every one.

In [ ]:
forms, truths = lab.make_forms(10, level="decent")
lab.show(forms[0])

## 3. Read all ten

One model call per form, with the plain prompt. About thirty seconds.

In [ ]:
answers = lab.read_all(forms, "plain")
print(lab.compare_one(answers[0], truths[0]))

## 4. How often is it wrong?

One number. This is the number a spec needs and an opinion does not have.

In [ ]:
print(lab.count_wrong(answers, truths))

## 5. Wrong **where**?

A rate on its own tells you to worry. It does not tell you what to do. Errors are never
spread evenly, and where they land decides everything.

In [ ]:
print(lab.by_field(answers, truths))

Look at which fields survived and which did not.

The model reads **words** reliably and misreads **codes**. Site names and disciplines
come through; work order numbers, equipment tags and permit numbers do not. That is not
a coincidence — a word has redundancy and a code has none, so one wrong character in
`PTW-48213` is undetectable and one wrong character in `Mechanical` is obvious.

## 6. Your rule

You cannot check 60 fields by hand for every batch. Pick the ones a person must see.

**Edit the list below** and re-run this cell and the next.

In [ ]:
CHECKED_BY_A_PERSON = ["permit"]      # <- your decision. Start here, then widen it.

print("a person checks:", CHECKED_BY_A_PERSON)
print("straight through:", [k for k in lab.FIELD_KEYS if k not in CHECKED_BY_A_PERSON])

## 7. What your rule costs — and what it still lets through

Two numbers. The second is the one nobody expects.

In [ ]:
print(lab.cost(CHECKED_BY_A_PERSON, answers, truths))

**Every field you leave unchecked is a field where a wrong value can reach the system
of record** — and you have just measured how many do.

Now widen the rule. Add `"equipment"` to the list above, re-run both cells, and watch
the two numbers move in opposite directions: the review cost goes up, the residual risk
goes down. Keep going and you can drive the risk to zero — by checking all six fields,
which is the same as not automating at all.

That dial is the decision. Not *"is the model good enough"*, which has no answer, but
*"how much human time am I buying how much safety with"*, which does.

## 8. The sentence you take to the capstone

Change `use_case` to your own brief and run it. This is the output of the lab.

In [ ]:
print(lab.capstone_sentence(CHECKED_BY_A_PERSON, answers, truths,
                            use_case="ticket to work order"))

## 9. What you take away

**A rate is not a decision.** "8 of 60 wrong" tells you to worry. "A person checks
permit and equipment, 20 reviews per 60 fields, 1 wrong value still gets through" tells
you what to build, and what you are accepting.

**The errors were all in the identifiers.** That is the general result, not an accident
of this form: codes have no redundancy, so a model's guess is undetectable. Wherever
your use case carries a reference number, a tag or a date, that is where the human goes.

**You measured it on documents you already knew the answers to.** Ten was enough to
see the shape. That is the cheapest useful thing you can do before trusting any
extraction — and it is the same move as Day 2's held-out set, applied to a new problem.

### Take this to Thursday

The sentence from section 8 goes in two places:

| Where | What it answers |
|---|---|
| **S27** deployment checklist, row 3 | *a wrong answer has a defined path* — yours is a review queue, and you can say how big |
| **S29** governance pack, template 4 | *approval points* — which fields need one, with the measurement behind the choice |

In [ ]:
print("09a done - you have a rule, its cost, and its residual risk.")